In [8]:
import numpy as np
import pandas as pd
import piplite
await piplite.install("openpyxl")
from openpyxl import load_workbook
await piplite.install("seaborn")
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
pd.pandas.set_option('display.max_columns', None)

In [9]:
df = pd.read_excel('Linear_Regression_NBA.xlsx')
df.head()
df.tail()

,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,draft_number,gp,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
311,Jarrett Allen,CLE,22,211,110,Texas,USA,2017,1,22,63,15.6,12.2,2.0,10.5,0.116,0.263,0.166,0.661,0.089,2020_21
312,Jaren Jackson Jr.,MEM,19,208,109,Michigon State,USA,2018,1,4,58,19.0,6.5,1.5,-2.3,0.053,0.150,0.228,0.591,0.072,2018_19
313,Jaren Jackson Jr.,MEM,20,208,109,Michigon State,USA,2018,1,4,57,22.0,5.8,1.7,-2.6,0.036,0.132,0.240,0.593,0.072,2019_20
314,Jaren Jackson Jr.,MEM,21,208,109,Michigon State,USA,2018,1,4,11,22.0,8.7,1.7,-10.5,0.069,0.187,0.259,0.551,0.068,2020_21
315,Jaren Jackson Jr.,MEM,22,208,109,Michigon State,USA,2018,1,4,78,21.5,7.7,1.5,7.9,0.057,0.169,0.254,0.535,0.058,2021_22


In [10]:
# get the every unique player according to name,weight and height
def code_player(df):
    unique_values_player = df[["player_name","player_height","player_weight"]].drop_duplicates()
    unique_values_player = unique_values_player.reset_index(drop=True)
    unique_values_player.reset_index(inplace=True)
    unique_values_player.rename(columns={"index":"ID"},inplace=True)
    return unique_values_player
# connect the dataframe by the player
def clean_data(df,unique_values_player,history_num=3):
    merged_df = pd.merge(unique_values_player, df, on=["player_name","player_height","player_weight"])
    merged_df.drop(columns=["player_height","player_weight","draft_year","draft_round","draft_number"], inplace=True)
    merged_df.drop(columns=["team_abbreviation","college","country"],inplace=True)
    filtered_df = merged_df.groupby('ID').filter(lambda x:len(x)>=history_num)
    filtered_df.sort_values(by=["ID","season"],ascending=[True,True])
    filtered_df.drop(columns=["season"],inplace=True)
    filtered_df["season_no"] = filtered_df.groupby("ID").cumcount()+1
    filtered_df["total_season"] = filtered_df.groupby("ID")["ID"].transform("count")
    dataset = filtered_df.reset_index(drop=True)
    return dataset
unique_values_player = code_player(df)
dataset = clean_data(df,unique_values_player)

# augment data
def data_augmentation(data,subset_rows=5):
    subblocks = []
    cnt = 0
    subset_rows = 5
    for attribute,group in dataset.groupby("ID"):
        len_group = len(group)
        group = group.reset_index(drop=True)
        if len_group > subset_rows:
            for i in range(len_group-subset_rows+1):
                subgroup = group.loc[i:i+subset_rows-1].copy()
                subgroup["block_id"] = cnt
                subblocks.append(subgroup)
                cnt += 1
        else:       
            group["block_id"] = cnt
            subblocks.append(group)
            cnt += 1
    augmented_data = pd.concat(subblocks).reset_index(drop=True)
    return augmented_data

# augmented_data = data_augmentation(dataset,5)

# splite data
def splite_data(data, column):
    test_players = [
        'Shai Gileous-Alexander', 'Dejounte Murray', 'Trae Young', 'Luka Doncic', 'Fred VanVleet',
        "De'Aaron Fox", 'Jalen Brunson', 'Markelle Fultz', 'Frank Ntilikina', 'Dennis Smith Jr.',
        'Monte Morris', "De'Anthony Melton", "Devonte' Graham", 'Jevon Carter', 'Aaron Holiday',
        'Collin Sexton', 'Jaylen Brown', 'Donovan Mitchell', 'Jayson Tatum', 'Brandon Ingram',
        'Lauri Markkanen', 'Bam Adebayo', 'Domantas Sabonis', 'Jarrett Allen', 'Jaren Jackson Jr.'
    ]

    # 從 block ID or ID 對應回 player_name
    id_name_map = data[["player_name", column]].drop_duplicates()

    # 根據 player_name 過濾出 test ids
    test_ids = id_name_map[id_name_map["player_name"].isin(test_players)][column].unique()
    train_ids = id_name_map[~id_name_map["player_name"].isin(test_players)][column].unique()

    train_set = data[data[column].isin(train_ids)]
    test_set = data[data[column].isin(test_ids)]

    train_set.to_excel(f'./dataset/trainset_{column}.xlsx', index=False)
    test_set.to_excel(f'./dataset/testset_{column}.xlsx', index=False)

def get_X(df):
    return pd.DataFrame(df[0:-1])
def get_y(df):
    return pd.DataFrame(df[len(df)-1:len(df)])
def load_data(column):
    train_data = pd.read_excel(f"./dataset/trainset_{column}.xlsx",index_col=False)
    test_data = pd.read_excel(f"./dataset/testset_{column}.xlsx",index_col=False)
    X_train = train_data.groupby(column).apply(lambda x:get_X(x)).reset_index(drop=True)
    y_train = train_data.groupby(column).apply(lambda x:get_y(x)).reset_index(drop=True)
    X_test = test_data.groupby(column).apply(lambda x:get_X(x)).reset_index(drop=True)
    y_test = test_data.groupby(column).apply(lambda x:get_y(x)).reset_index(drop=True)
    return X_train,y_train,X_test,y_test

def build_features(df):
    age = df.age.values[-1]
    gp_mean = np.mean(df.gp.values)
    pts_mean = np.mean(df.pts.values)
    pts_last1year= df.pts.values[-1]
    pts_last2year = df.pts.values[-2]
    net_mean = np.mean(df.net_rating.values)
    ts_mean = np.mean(df.ts_pct.values)
    usg_mean = np.mean(df.usg_pct.values)
    n_season = df['season_no'].values[-1]
    data = np.array([[age, gp_mean, pts_mean, pts_last1year, pts_last2year, net_mean,ts_mean, usg_mean, n_season]])
    return pd.DataFrame(data)

def extract_feature(column,X_train,y_train,X_test,y_test):
    df_feature_train = X_train.sort_values([column],ascending=[True]).groupby(column).apply(lambda x: build_features(x))
    x_feature_train = df_feature_train.values
    labels_train = y_train.sort_values([column],ascending=[True]).pts.values

    df_feature_test = X_test.sort_values([column],ascending=[True]).groupby(column).apply(lambda x: build_features(x))
    x_feature_test = df_feature_test.values
    labels_test = y_test.sort_values([column],ascending=[True]).pts.values
    return x_feature_train,labels_train,x_feature_test,labels_test

def process_raw_data(df,min_num=1,block_size=3):
    unique_values_player = code_player(df)
    dataset = clean_data(df,unique_values_player,min_num)
    augmented_data = data_augmentation(dataset,block_size)
    splite_data(augmented_data,"block_id")
    splite_data(dataset,"ID")

def get_processed_data(column):
    x_feature_train,labels_train,x_feature_test,labels_test = extract_feature(column,*load_data(column))
    
    scaler = StandardScaler()
    scaler.fit(x_feature_train)
    x_feature_train = scaler.transform(x_feature_train)
    x_feature_test = scaler.transform(x_feature_test)
    return x_feature_train,labels_train,x_feature_test,labels_test

def data_pipeline(df,is_augmented,min_num = 1,block_size = 3):
    if is_augmented:
        column = "block_id"
    else:
        column = "ID"
    process_raw_data(df,min_num=1,block_size=3)
    x_feature_train,labels_train,x_feature_test,labels_test = get_processed_data(column)
    return x_feature_train,labels_train,x_feature_test,labels_test

x_feature_train,labels_train,x_feature_test,labels_test = data_pipeline(df,0,block_size=3)
lr = LinearRegression()
lr.fit(x_feature_train, labels_train)
y_pred_test = lr.predict(x_feature_test)
y_pred_train = lr.predict(x_feature_train)
mse_test = mean_squared_error(labels_test[:16], y_pred_test[:16])
mse_train = mean_squared_error(labels_train, y_pred_train)
#print(mse_test)
#print(mse_train)

<ipython-input-10-402098e2f866>:76: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  X_train = train_data.groupby(column).apply(lambda x:get_X(x)).reset_index(drop=True)
<ipython-input-10-402098e2f866>:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y_train = train_data.groupby(column).apply(lambda x:get_y(x)).reset_index(drop=True)
<ipython-input-10-402098e2f866>:78: DeprecationWarning: DataFrameGroupBy.app

In [11]:
# 假設你現在是用 "ID" 分群的（沒用 block_id augmentation）
column = "ID"  # 如果你改成 block_id 就換成 "block_id"

# 讀回原本的測試集（含 player_name）
test_data = pd.read_excel(f"./dataset/testset_{column}.xlsx")

# 抓出每個 group（ID 或 block_id）中最後一筆資料（對應到 y_test）
name_list = test_data.groupby(column).tail(1)["player_name"].values

# 建出結果 DataFrame
result_df = pd.DataFrame({
    "Player": name_list,
    "Actual": labels_test,
    "Predicted": y_pred_test.flatten(),
    "Error": y_pred_test.flatten() - labels_test
})

# 顯示結果
print(result_df.head(16))

# 計算並印出 MAPE / MSE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape_test = mean_absolute_percentage_error(labels_test[:16], y_pred_test[:16])
mape_train = mean_absolute_percentage_error(labels_train, y_pred_train)

print(f"Test MSE: {mse_test:.3f}")
print(f"Train MSE: {mse_train:.3f}")
print(f"Test MAPE of all PG: {mape_test:.2f}%")
print(f"Train MAPE: {mape_train:.2f}%")


                    Player  Actual  Predicted     Error
0   Shai Gileous-Alexander    25.4  24.593151 -0.806849
1          Dejounte Murray    17.7  18.067180  0.367180
2               Trae Young    29.3  27.710480 -1.589520
3              Luka Doncic    28.9  31.569239  2.669239
4            Fred VanVleet    17.8  19.077126  1.277126
5             De'Aaron Fox    25.8  24.190875 -1.609125
6            Jalen Brunson    18.3  20.967134  2.667134
7           Markelle Fultz    17.2  20.689750  3.489750
8          Frank Ntilikina    10.0  17.102413  7.102413
9         Dennis Smith Jr.    13.2  17.786268  4.586268
10            Monte Morris    14.5  17.740990  3.240990
11       De'Anthony Melton    17.2  17.363873  0.163873
12         Devonte' Graham    15.0  19.206156  4.206156
13            Jevon Carter    11.0  16.910551  5.910551
14           Aaron Holiday    14.1  19.781609  5.681609
15           Collin Sexton    20.1  25.035163  4.935163
Test MSE: 14.076
Train MSE: 5.997
Test MAPE of a

In [12]:
mape_test = mean_absolute_percentage_error(labels_test[:7], y_pred_test[:7])
print(f"Test MAPE of all star: {mape_test:.2f}%")

Test MAPE of all star: 6.84%


In [13]:
mape_test = mean_absolute_percentage_error(labels_test[7:16], y_pred_test[7:16])
print(f"Test MAPE of role player: {mape_test:.2f}%")

Test MAPE of role player: 32.89%


In [14]:
svr = SVR(kernel='rbf',C=100,gamma=0.01,epsilon=0.2)
svr.fit(x_feature_train, labels_train)
y_pred_test = svr.predict(x_feature_test)
y_pred_train = svr.predict(x_feature_train)
mse_test = mean_squared_error(labels_test[:16], y_pred_test[:16])
mse_train = mean_squared_error(labels_train, y_pred_train)

In [15]:
result_df = pd.DataFrame({
    "Player": name_list,
    "Actual": labels_test,
    "Predicted": y_pred_test.flatten(),
    "Error": y_pred_test.flatten() - labels_test
})

# 顯示結果
print(result_df.head(16))

# 計算並印出 MAPE / MSE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape_test = mean_absolute_percentage_error(labels_test[:16], y_pred_test[:16])
mape_train = mean_absolute_percentage_error(labels_train, y_pred_train)

print(f"Test MSE: {mse_test:.3f}")
print(f"Train MSE: {mse_train:.3f}")
print(f"Test MAPE of all PG: {mape_test:.2f}%")
print(f"Train MAPE: {mape_train:.2f}%")


                    Player  Actual  Predicted     Error
0   Shai Gileous-Alexander    25.4  24.842161 -0.557839
1          Dejounte Murray    17.7  16.624446 -1.075554
2               Trae Young    29.3  27.501277 -1.798723
3              Luka Doncic    28.9  27.793539 -1.106461
4            Fred VanVleet    17.8  17.040405 -0.759595
5             De'Aaron Fox    25.8  22.413596 -3.386404
6            Jalen Brunson    18.3  19.893590  1.593590
7           Markelle Fultz    17.2  17.556462  0.356462
8          Frank Ntilikina    10.0  14.475057  4.475057
9         Dennis Smith Jr.    13.2  15.481548  2.281548
10            Monte Morris    14.5  15.991231  1.491231
11       De'Anthony Melton    17.2  16.336626 -0.863374
12         Devonte' Graham    15.0  16.256072  1.256072
13            Jevon Carter    11.0  14.679730  3.679730
14           Aaron Holiday    14.1  16.700993  2.600993
15           Collin Sexton    20.1  22.889672  2.789672
Test MSE: 4.907
Train MSE: 4.211
Test MAPE of al

In [16]:
mape_test = mean_absolute_percentage_error(labels_test[:7], y_pred_test[:7])
print(f"Test MAPE of all star: {mape_test:.2f}%")

Test MAPE of all star: 6.33%


In [17]:
mape_test = mean_absolute_percentage_error(labels_test[7:16], y_pred_test[7:16])
print(f"Test MAPE of role player: {mape_test:.2f}%")

Test MAPE of role player: 17.06%
